In [1]:
from load_data import load_dataset

plos = load_dataset("plos")
elife = load_dataset("elife")
plos.head()

,dataset,split,id,abstract,summary,n_abstract_sents,n_summary_sents
0,plos,train,journal.pgen.0030189,"Kidney function depends on the nephron , which...","In the kidney , structures known as nephrons a...",10,9
1,plos,train,journal.ppat.1006076,White-nose syndrome is one of the most lethal ...,Many species of bats in North America have bee...,9,5
2,plos,train,journal.pntd.0007498,Sustainable dengue intervention requires the p...,The burden of dengue has been increasing over ...,11,12
3,plos,train,journal.pgen.1001012,Despite the central role of estrogen exposure ...,Estrogen exposure is the most important risk f...,11,7
4,plos,train,journal.pntd.0000605,Macrophage migration inhibitory factor ( MIF )...,Melioidosis is a severe tropical infection cau...,13,9


In [2]:
#Sanity check does sentence count and word counts match
import pandas as pd

def table1_stats(df):
    return pd.Series({
        "n_docs": len(df),
        "summary_words": df["summary"].str.split().str.len().mean(),
        "summary_sents": df["n_summary_sents"].mean(),
    })

pd.DataFrame({"PLOS": table1_stats(plos), "eLife": table1_stats(elife)}).round(1)

,PLOS,eLife
n_docs,27525.0,4828.0
summary_words,194.9,382.7
summary_sents,7.8,15.7


In [3]:
#Counting only tokens that contain at least one letter or digit
#This gets closer and is almost identical to the paper
import re

def count_words(text):
    return sum(1 for tok in text.split() if re.search(r"\w", tok))

for name, df in [("PLOS", plos), ("eLife", elife)]:
    print(name, round(df["summary"].apply(count_words).mean(), 1))

PLOS 176.6
eLife 348.4


In [4]:
import pandas as pd

combined = pd.concat([plos, elife], ignore_index=True)

long = combined.melt(
    id_vars=["dataset", "split", "id"],
    value_vars=["abstract", "summary"],
    var_name="text_type",
    value_name="text",
)

print(long.shape)  # expect (64706, 5)
long.groupby(["dataset", "text_type"]).size()

(64706, 5)


dataset  text_type
elife    abstract      4828
         summary       4828
plos     abstract     27525
         summary      27525
dtype: int64

In [6]:
import time
import textstat

print(textstat.__version__)  # log this; scores can differ between versions

# 1. Try it on a sample and time it
sample = long.sample(200, random_state=42)
t0 = time.time()
sample_scores = sample["text"].apply(textstat.flesch_kincaid_grade)
elapsed = time.time() - t0
print(f"{elapsed:.2f}s for 200 texts -> ~{elapsed / 200 * len(long) / 60:.1f} min for all")
print(sample_scores.describe())

(0, 7, 2)
0.40s for 200 texts -> ~2.2 min for all
count    200.000000
mean      14.412000
std        2.619938
min        7.000000
25%       12.800000
50%       14.600000
75%       16.225000
max       22.100000
Name: text, dtype: float64


In [7]:
long["fkgl"] = long["text"].apply(textstat.flesch_kincaid_grade)


In [ ]:
#New version of textstat
long.groupby(["dataset", "text_type"])["fkgl"].agg(["mean", "std"]).round(2)

mean   std
dataset text_type             
elife   abstract   16.69  2.28
        summary    11.86  1.47
plos    abstract   16.16  2.60
        summary    15.94  2.34

In [9]:
long["words_per_sent"] = long["text"].apply(textstat.avg_sentence_length)
long["syll_per_word"] = long["text"].apply(textstat.avg_syllables_per_word)
long.groupby(["dataset", "text_type"])[["words_per_sent", "syll_per_word"]].mean().round(3)

words_per_sent  syll_per_word
dataset text_type                               
elife   abstract           21.964          2.010
        summary            19.727          1.674
plos    abstract           22.509          1.947
        summary            22.674          1.923

In [7]:
#Installed an older version of textstat to see if the discrepancy is purely because of differing versions
#pip install "textstat==0.7.3"
#Numbers match exactly now
long["fkgl"] = long["text"].apply(textstat.flesch_kincaid_grade)
long.groupby(["dataset", "text_type"])["fkgl"].agg(["mean", "std"]).round(2)

mean   std
dataset text_type             
elife   abstract   15.57  2.24
        summary    10.92  1.45
plos    abstract   15.04  2.59
        summary    14.76  2.33

In [ ]:
#Coleman Liau Index
long["cli"] = long["text"].apply(textstat.coleman_liau_index)

long.groupby(["dataset", "text_type"])["cli"].agg(["mean", "std"]).round(2)

mean   std
dataset text_type             
elife   abstract   17.68  2.27
        summary    12.51  1.35
plos    abstract   16.39  2.24
        summary    15.90  2.01

In [9]:
#Dale Chall Readability Score
long["dcrs"] = long["text"].apply(textstat.dale_chall_readability_score)
long.groupby(["dataset", "text_type"])["dcrs"].agg(["mean", "std"]).round(2)

mean   std
dataset text_type             
elife   abstract   11.77  0.85
        summary     8.93  0.53
plos    abstract   11.06  0.86
        summary    10.91  0.85